In [1]:
from uuid import uuid4
import os
import pandas as pd
from databricks.sdk import WorkspaceClient

# https://github.com/AgDMALabs-Public/ag-vision-dataops
from ag_vision.rover.ingest import RoverDataIngest

# Connect to Roboflow 

In [2]:
w = WorkspaceClient(profile='agpile')  # set your profile
w.config.host

'https://dbc-0f3d94f2-e27b.cloud.databricks.com'

# Define the local and Databricks Variables

In [3]:
SCAN_DATE = '11/25/2025'

YEAR = 2025
COUNTRY = 'IND' # Three letter coutry code.
CROP = 'maize'
CROP_GROWTH_STAGE = 'VT'
SEASON = 'spring'

TRIAL_NAME = 'BETS'
SITE = 'TNAU-Coimbatore'
FIELD = 'West_Field_03'
LOCATION = 'Section_B'

ROVER_LOCAL_PATH = '/Users/danielwilliams/Documents/Field Data/rover_upload_test'
DB_PROJECT_DIR = '/Volumes/ue1_prod_catalog_119738067017277/tier1_raw/data'

raw_image_dir = f'{ROVER_LOCAL_PATH}/raw_data/'
stiched_image_dir = f'{ROVER_LOCAL_PATH}/stiched_data/'
plot_image_dir = f'{ROVER_LOCAL_PATH}/plot_data/'

plot_boundary_key = f"{ROVER_LOCAL_PATH}/plot_boundary.geojson"
metadata_key = f"{ROVER_LOCAL_PATH}/rover_details.json"

CAMERA = 'rgb'
SCAN_DATE = '1/2/2026'
STICHED_DATE = '1/2/2026'
STICHING_METHOD = 'gemini'
PLOT_CROP_DATE = '1/2/2026'

# Define the Metadata

In [4]:
rover_metadata = {
    "id": str(uuid4()),
    "name": "stand_count",  # this will be the mission name in the folder path
    "task": "rover_data_collection",
    "location": {
        "site": SITE.lower(),  # Try to keep these standard ORG-Site EX CIAT-Cali, or CIAT-Arusha....
        "field": FIELD.lower(),
        # what is the name of the field that the trial was run on? The field and location can be the same name if there is no difference.
        "location": LOCATION.lower()
        # The location corresponds to a specific field book. If data is in EBS match the location name.
    },
    "trialProperties": {
        "name": TRIAL_NAME  # What is the name of the trial, a trial usually has multiple locations.
    },
    "rover_acquisition_properties": {
        "date": SCAN_DATE,
        "rover_make": "NewCo",
        "rover_model": "Rover-1",
        "camera_make": "sony",
        "camera_model": "MX-1000",
        "cameraHeight": 45.5,
        "horizontalOverlapPercentage": 75.0,
        "verticalOverlapPercentage": 70.0,
        "gpsQuality": "RTK Fixed"
    },
    "agronomic_properties": {
        "crop_type": CROP,  # Required
        "growth_stage": CROP_GROWTH_STAGE,  # optional
        "soil_color": None,
        "weed_pressure": None,
        "irrigation_level": None,
        "tillage_type": None,
        "fertilizer_level": None
    }
}

# List out the Raw Images.

In [5]:
raw_files = os.listdir(raw_image_dir)
raw_files = [raw_image_dir + x for x in raw_files]

print(f"there are {len(raw_files)} raw files in the directory")

there are 15 raw files in the directory


# List out the Stiched Images

In [6]:
stiched_files = os.listdir(stiched_image_dir)
stiched_files = [stiched_image_dir + x for x in stiched_files]

print(f"there are {len(stiched_files)} raw files in the directory")

stiched_df = pd.DataFrame({'src_path': stiched_files})
stiched_df['camera'] = CAMERA
stiched_df['method'] = STICHING_METHOD
stiched_df['stiched_date'] = STICHED_DATE

there are 15 raw files in the directory


# List out the plot Images

In [7]:
plot_files = os.listdir(plot_image_dir)
plot_files = [plot_image_dir + x for x in plot_files]

print(f"there are {len(plot_files)} raw files in the directory")

plot_df = pd.DataFrame({'src_path': plot_files})
plot_df['camera'] = CAMERA
plot_df['file_generation_datetime'] = PLOT_CROP_DATE

there are 15 raw files in the directory


# Logic to add plot ID to the plot_df

In [8]:
# This will need to be custom on how the data is stored.
plot_df['plot_id'] = plot_df['src_path'].apply(lambda x: os.path.splitext(os.path.basename(x))[0])

# Start the Ingest

In [9]:
# This points to the aps1-prod-tnau-fg workspace _YOU SHOULD NOT NEED TO CHANGE THIS.
# TNAU
#workspace_bucket = '/Volumes/aps1_prod_tnau_fg_catalog_1336582592012881/tier1_raw/data'
# AgPile
ingest = RoverDataIngest(platform='local',  # DONT CHANGE THIS
                         cloud_bucket=DB_PROJECT_DIR,  # Set this to the bucket you want to save the data to.
                         cloud_client=w,  # should not need to change.
                         scan_date=SCAN_DATE,
                         plot_boundary_key=plot_boundary_key,
                         scan_metadata_key=metadata_key)


In [10]:
ingest.load_metadata_from_dict(metadata_dict=rover_metadata)

# season is needed to generate the mission dir, This has lots of Validation and will throw assert errors.
ingest.add_season_code_to_metadata(year=YEAR,
                                   country=COUNTRY,
                                   crop=CROP,
                                   time_of_year=TIME_OF_YEAR)

# This is the main dir where all the data will be stored.
ingest.generate_rover_mission_dir_path()

In [11]:
# Save the Metadata
ingest.save_metadata_to_json_local()
ingest.upload_metadata_to_db()

Uploading: 100%|██████████| 1.40k/1.40k [00:00<00:00, 1.93kB/s]


In [12]:
ingest.upload_scan_plot_boundary_to_db()

Saving to /Volumes/ue1_prod_catalog_119738067017277/tier1_raw/data/tnau-coimbatore/bets/2025:ind:maize:spring/west_field_03/section_b/rover/stand_count/field_data/plot_boundary.geojson


Uploading: 100%|██████████| 190k/190k [00:00<00:00, 352kB/s] 


In [13]:
ingest.rover_mission_dir

'/Volumes/ue1_prod_catalog_119738067017277/tier1_raw/data/tnau-coimbatore/bets/2025:ind:maize:spring/west_field_03/section_b/rover/stand_count'

# Ingest the Raw Data

In [14]:
# This will be a list of all the raw files from the flight eg: nav, bin, tif, and jpg files.
ingest.generate_raw_ingest_df(file_list=raw_files,
                              camera=CAMERA)


In [15]:
ingest.generate_raw_image_dst_path_name()

In [16]:
ingest.upload_raw_scan_data_to_db()

0it [00:00, ?it/s]
Uploading:   0%|          | 0.00/7.43M [00:00<?, ?B/s]
Uploading:   4%|▍         | 304k/7.08M [00:00<00:02, 3.07MB/s]
Uploading:   8%|▊         | 608k/7.08M [00:00<00:02, 2.95MB/s]
Uploading:  16%|█▌        | 1.12M/7.08M [00:00<00:01, 3.67MB/s]
Uploading:  24%|██▍       | 1.72M/7.08M [00:00<00:01, 4.18MB/s]
Uploading:  32%|███▏      | 2.23M/7.08M [00:00<00:01, 4.55MB/s]
Uploading:  43%|████▎     | 3.08M/7.08M [00:00<00:00, 5.16MB/s]
Uploading:  50%|█████     | 3.56M/7.08M [00:00<00:00, 4.64MB/s]
Uploading:  57%|█████▋    | 4.02M/7.08M [00:01<00:00, 3.99MB/s]
Uploading:  62%|██████▏   | 4.41M/7.08M [00:01<00:00, 3.52MB/s]
Uploading:  68%|██████▊   | 4.80M/7.08M [00:01<00:00, 3.53MB/s]
Uploading:  73%|███████▎  | 5.14M/7.08M [00:01<00:00, 3.31MB/s]
Uploading:  77%|███████▋  | 5.47M/7.08M [00:01<00:00, 3.19MB/s]
Uploading:  82%|████████▏ | 5.78M/7.08M [00:01<00:00, 2.88MB/s]
Uploading:  86%|████████▌ | 6.06M/7.08M [00:01<00:00, 2.75MB/s]
Uploading:  89%|████████▉ | 6.33

# Ingest the Stiched Data

In [17]:
ingest.generate_stiched_ingest_df(df=stiched_df,
                                  generate_uuid=True) # This will give the files new UUIDS for the image names. Set to False if you don't want to change it.

In [18]:
ingest.generate_stiched_image_dst_path_name()

In [19]:
ingest.upload_stiched_images_to_db()

0it [00:00, ?it/s]
Uploading:   0%|          | 0.00/7.43M [00:00<?, ?B/s]
Uploading:   6%|▌         | 448k/7.08M [00:00<00:01, 3.75MB/s]
Uploading:  11%|█▏        | 816k/7.08M [00:00<00:02, 2.87MB/s]
Uploading:  15%|█▌        | 1.09M/7.08M [00:00<00:02, 2.45MB/s]
Uploading:  19%|█▉        | 1.34M/7.08M [00:00<00:02, 2.34MB/s]
Uploading:  22%|██▏       | 1.58M/7.08M [00:00<00:02, 2.37MB/s]
Uploading:  26%|██▌       | 1.81M/7.08M [00:00<00:02, 2.35MB/s]
Uploading:  29%|██▉       | 2.05M/7.08M [00:00<00:02, 2.32MB/s]
Uploading:  32%|███▏      | 2.30M/7.08M [00:00<00:02, 2.35MB/s]
Uploading:  36%|███▌      | 2.53M/7.08M [00:01<00:02, 2.24MB/s]
Uploading:  39%|███▉      | 2.77M/7.08M [00:01<00:01, 2.30MB/s]
Uploading:  42%|████▏     | 3.00M/7.08M [00:01<00:01, 2.32MB/s]
Uploading:  46%|████▌     | 3.23M/7.08M [00:01<00:01, 2.18MB/s]
Uploading:  49%|████▉     | 3.47M/7.08M [00:01<00:01, 2.19MB/s]
Uploading:  52%|█████▏    | 3.70M/7.08M [00:01<00:01, 2.14MB/s]
Uploading:  56%|█████▌    | 3.97

# Ingest the Plot Images

In [20]:
ingest.generate_plot_ingest_df(df=plot_df,
                               generate_uuid=True) # This will give the files new UUIDS for the image names. Set to False if you dont want to change it.

In [21]:
ingest.generate_plot_image_dst_path_name()

In [22]:
ingest.upload_plot_images_to_db()

0it [00:00, ?it/s]
Uploading:   0%|          | 0.00/7.43M [00:00<?, ?B/s]
Uploading:   8%|▊         | 576k/7.08M [00:00<00:01, 5.03MB/s]
Uploading:  15%|█▍        | 1.05M/7.08M [00:00<00:01, 3.28MB/s]
Uploading:  20%|█▉        | 1.39M/7.08M [00:00<00:02, 2.94MB/s]
Uploading:  24%|██▍       | 1.69M/7.08M [00:00<00:02, 2.76MB/s]
Uploading:  28%|██▊       | 1.97M/7.08M [00:00<00:01, 2.72MB/s]
Uploading:  32%|███▏      | 2.23M/7.08M [00:00<00:01, 2.70MB/s]
Uploading:  35%|███▌      | 2.50M/7.08M [00:00<00:02, 2.39MB/s]
Uploading:  39%|███▉      | 2.78M/7.08M [00:01<00:01, 2.48MB/s]
Uploading:  43%|████▎     | 3.03M/7.08M [00:01<00:02, 2.12MB/s]
Uploading:  46%|████▌     | 3.25M/7.08M [00:01<00:01, 2.04MB/s]
Uploading:  50%|████▉     | 3.53M/7.08M [00:01<00:01, 2.23MB/s]
Uploading:  53%|█████▎    | 3.78M/7.08M [00:01<00:01, 2.30MB/s]
Uploading:  57%|█████▋    | 4.03M/7.08M [00:01<00:01, 2.35MB/s]
Uploading:  61%|██████    | 4.31M/7.08M [00:01<00:01, 2.45MB/s]
Uploading:  64%|██████▍   | 4.5